In [1]:
from pathlib import Path
import sys
import importlib
import zipfile

import joblib
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor


CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (path / "src" / "feature_engineering.py").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Не найден корень проекта.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.feature_engineering as fe
importlib.reload(fe)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
SUBMISSION_DIR = PROJECT_ROOT / "submission"

train = pd.read_parquet(
    PROCESSED_DIR / "train_canonical.parquet"
)

test_raw = pd.read_csv(
    PROJECT_ROOT / "train_data" / "X_test_base.csv"
)

TARGET_COLUMN = "Цена"

X_train_raw = train.drop(columns=[TARGET_COLUMN]).copy()
y = train[TARGET_COLUMN].copy()

X_train_features = fe.add_title_hierarchy_features(
    fe.prepare_features(X_train_raw)
)

X_test_features = fe.add_title_hierarchy_features(
    fe.prepare_features(test_raw)
)

print("Train features:", X_train_features.shape)
print("Test features:", X_test_features.shape)

Train features: (8340, 66)
Test features: (8341, 66)


Подготовить v7 и обучить только отсутствующую final v7

In [2]:
RAW_DUPLICATE_NUMERIC_COLUMNS = [
    "Пробег",
    "Расход",
    "Количество цилиндров",
    "Двери",
    "Количество кресел",
]

EXCLUDED_COLUMNS = [
    "car_id",
    "Предложение",
] + RAW_DUPLICATE_NUMERIC_COLUMNS

EXCLUDED_COLUMNS_V7 = EXCLUDED_COLUMNS + [
    "Полное название",
    "Цвет",
]

feature_columns_v7 = [
    column
    for column in X_train_features.columns
    if column not in EXCLUDED_COLUMNS_V7
]

X_train_v7 = X_train_features[
    feature_columns_v7
].copy()

X_test_v7 = X_test_features[
    feature_columns_v7
].copy()

numeric_columns_v7 = X_train_v7.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_columns_v7 = [
    column
    for column in feature_columns_v7
    if column not in numeric_columns_v7
]


def coerce_v7_frame(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    result = frame.copy()

    for column in numeric_columns_v7:
        result[column] = pd.to_numeric(
            result[column],
            errors="coerce",
        ).astype(float)

    for column in categorical_columns_v7:
        result[column] = (
            result[column]
            .astype("string")
            .fillna("__MISSING__")
            .astype(str)
        )

    return result


X_train_v7 = coerce_v7_frame(X_train_v7)
X_test_v7 = coerce_v7_frame(X_test_v7)

assert X_train_v7.shape[1] == 57
assert X_test_v7.shape[1] == 57
assert len(X_test_v7) == 8341

In [3]:
V7_FINAL_PATH = (
    MODELS_DIR
    / "catboost_title_hierarchy_v7_final_3000.cbm"
)

v7_final_model = CatBoostRegressor()

if V7_FINAL_PATH.exists():
    v7_final_model.load_model(V7_FINAL_PATH)
    print("Loaded existing final v7 model.")
else:
    v7_final_model = CatBoostRegressor(
        loss_function="RMSE",
        iterations=3000,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=5,
        random_seed=42,
        verbose=500,
        allow_writing_files=False,
    )

    v7_final_model.fit(
        X_train_v7,
        np.log1p(y),
        cat_features=categorical_columns_v7,
    )

    v7_final_model.save_model(V7_FINAL_PATH)

    print("Saved:", V7_FINAL_PATH)

0:	learn: 0.6524812	total: 220ms	remaining: 11m
500:	learn: 0.1583883	total: 56.8s	remaining: 4m 43s
1000:	learn: 0.1217319	total: 2m 12s	remaining: 4m 23s
1500:	learn: 0.0999435	total: 3m 59s	remaining: 3m 59s
2000:	learn: 0.0848328	total: 5m 33s	remaining: 2m 46s
2500:	learn: 0.0736916	total: 6m 57s	remaining: 1m 23s
2999:	learn: 0.0648002	total: 8m 14s	remaining: 0us
Saved: C:\temp\shift_ml\models\catboost_title_hierarchy_v7_final_3000.cbm


Загрузить остальные пять готовых моделей

In [4]:
v5_final_model = CatBoostRegressor()
v5_final_model.load_model(
    MODELS_DIR
    / "catboost_title_hierarchy_v5_final_3000.cbm"
)

v6_final_model = CatBoostRegressor()
v6_final_model.load_model(
    MODELS_DIR
    / "catboost_title_hierarchy_v6_final_3000.cbm"
)

stats_final_model = CatBoostRegressor()
stats_final_model.load_model(
    MODELS_DIR
    / "catboost_v7_hierarchical_stats_v1_final.cbm"
)

ridge_final_model = joblib.load(
    MODELS_DIR
    / "ridge_log_target_alpha_0_1_final.joblib"
)

text_final_model = joblib.load(
    MODELS_DIR
    / "text_ridge_tfidf_alpha_1_0_final.joblib"
)

print("v5 features:", len(v5_final_model.feature_names_))
print("v6 features:", len(v6_final_model.feature_names_))
print("v7 features:", len(v7_final_model.feature_names_))
print("stats features:", len(stats_final_model.feature_names_))
print("Ridge type:", type(ridge_final_model).__name__)
print("Text type:", type(text_final_model).__name__)

v5 features: 59
v6 features: 58
v7 features: 57
stats features: 69
Ridge type: TransformedTargetRegressor
Text type: dict


. Построить test-признаки для CatBoost v5, v6 и v7

In [5]:
def make_catboost_input(
    features: pd.DataFrame,
    model: CatBoostRegressor,
    model_name: str,
) -> pd.DataFrame:
    expected_columns = list(model.feature_names_)

    missing_columns = [
        column
        for column in expected_columns
        if column not in features.columns
    ]

    if missing_columns:
        raise KeyError(
            f"{model_name}: отсутствуют признаки: {missing_columns}"
        )

    result = features[expected_columns].copy()

    cat_indices = model.get_cat_feature_indices()

    cat_columns = [
        expected_columns[index]
        for index in cat_indices
    ]

    numeric_columns = [
        column
        for column in expected_columns
        if column not in cat_columns
    ]

    for column in cat_columns:
        result[column] = (
            result[column]
            .astype("string")
            .fillna("__MISSING__")
            .astype(str)
        )

    for column in numeric_columns:
        result[column] = pd.to_numeric(
            result[column],
            errors="coerce",
        ).astype(float)

    assert result.columns.tolist() == expected_columns

    return result


X_test_v5 = make_catboost_input(
    features=X_test_features,
    model=v5_final_model,
    model_name="v5",
)

X_test_v6 = make_catboost_input(
    features=X_test_features,
    model=v6_final_model,
    model_name="v6",
)

X_test_v7_for_model = make_catboost_input(
    features=X_test_features,
    model=v7_final_model,
    model_name="v7",
)

print(X_test_v5.shape, X_test_v6.shape, X_test_v7_for_model.shape)

(8341, 59) (8341, 58) (8341, 57)


Построить target stats V1 для test

Это не обучение CatBoost. Здесь просто считаются 12 признаков по полному train, как и должно быть для test.

In [6]:
GROUP_SPECS_V1 = {
    "brand_model": [
        "Бренд",
        "Модель",
    ],
    "brand_model_year": [
        "Бренд",
        "Модель",
        "Год выпуска",
    ],
    "title_prefix3": [
        "Название_префикс_3",
    ],
    "title_prefix3_year": [
        "Название_префикс_3",
        "Год выпуска",
    ],
    "title_normalized": [
        "Название_нормализованное_без_года",
    ],
    "title_normalized_year": [
        "Название_нормализованное_без_года",
        "Год выпуска",
    ],
}


def make_group_key(
    frame: pd.DataFrame,
    columns: list[str],
) -> pd.Series:
    key = None

    for column in columns:
        series = frame[column]

        if pd.api.types.is_numeric_dtype(series):
            part = (
                pd.to_numeric(
                    series,
                    errors="coerce",
                )
                .round(4)
                .astype("Float64")
                .astype("string")
            )
        else:
            part = series.astype("string")

        part = (
            part
            .fillna("__MISSING__")
            .str.strip()
            .str.upper()
        )

        key = part if key is None else key.str.cat(
            part,
            sep="|||",
        )

    return key


def make_target_stats_for_test(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_apply: pd.DataFrame,
    group_specs: dict[str, list[str]],
    smoothing: float = 20.0,
) -> pd.DataFrame:
    target_log = np.log1p(
        y_train.to_numpy(dtype=float)
    )

    global_mean = float(target_log.mean())

    output = pd.DataFrame(
        index=X_apply.index,
    )

    for group_name, group_columns in group_specs.items():
        train_keys = make_group_key(
            X_train,
            group_columns,
        )

        apply_keys = make_group_key(
            X_apply,
            group_columns,
        )

        group_table = pd.DataFrame(
            {
                "key": train_keys.to_numpy(),
                "target_log": target_log,
            }
        )

        group_stats = (
            group_table
            .groupby("key", sort=False)
            .agg(
                count=("target_log", "size"),
                target_sum=("target_log", "sum"),
            )
        )

        group_stats["smooth_mean_log_price"] = (
            group_stats["target_sum"]
            + smoothing * global_mean
        ) / (
            group_stats["count"] + smoothing
        )

        apply_count = (
            apply_keys
            .map(group_stats["count"])
            .fillna(0)
            .to_numpy(dtype=float)
        )

        apply_smooth_mean = (
            apply_keys
            .map(group_stats["smooth_mean_log_price"])
            .fillna(global_mean)
            .to_numpy(dtype=float)
        )

        output[
            f"te_{group_name}_log_count"
        ] = np.log1p(apply_count)

        output[
            f"te_{group_name}_smooth_mean_log_price"
        ] = apply_smooth_mean

    return output


X_test_stats = make_target_stats_for_test(
    X_train=X_train_v7,
    y_train=y,
    X_apply=X_test_v7,
    group_specs=GROUP_SPECS_V1,
    smoothing=20.0,
)

X_test_stats_model = pd.concat(
    [
        X_test_v7,
        X_test_stats,
    ],
    axis=1,
)

X_test_stats_for_model = make_catboost_input(
    features=X_test_stats_model,
    model=stats_final_model,
    model_name="stats",
)

assert X_test_stats_for_model.shape[1] == 69

print(X_test_stats_for_model.shape)

(8341, 69)


Предсказания шести моделей

In [7]:
v5_test_pred = np.maximum(
    np.expm1(v5_final_model.predict(X_test_v5)),
    1,
)

v6_test_pred = np.maximum(
    np.expm1(v6_final_model.predict(X_test_v6)),
    1,
)

v7_test_pred = np.maximum(
    np.expm1(v7_final_model.predict(X_test_v7_for_model)),
    1,
)

stats_test_pred = np.maximum(
    np.expm1(
        stats_final_model.predict(
            X_test_stats_for_model
        )
    ),
    1,
)

Для Ridge и Text попробуем передать ровно те признаки, которые они запомнили при fit.

In [14]:
import zipfile

import numpy as np
import pandas as pd


def sanitize_sklearn_frame(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    """
    Убирает pandas pd.NA, потому что sklearn SimpleImputer
    и некоторые pipeline не умеют с ним работать.
    """
    result = frame.copy()

    for column in result.columns:
        series = result[column]

        if (
            pd.api.types.is_numeric_dtype(series)
            or pd.api.types.is_bool_dtype(series)
        ):
            result[column] = pd.to_numeric(
                series,
                errors="coerce",
            ).astype(float)

        else:
            result[column] = pd.Series(
                series.astype("string").to_numpy(
                    dtype=object,
                    na_value=np.nan,
                ),
                index=result.index,
                dtype=object,
            )

    return result


def find_feature_names_in(
    model,
) -> list[str] | None:
    """
    Ищет имена входных колонок внутри sklearn-объекта,
    включая TransformedTargetRegressor и Pipeline.
    """
    visited = set()

    def walk(object_):
        if object_ is None:
            return None

        object_id = id(object_)

        if object_id in visited:
            return None

        visited.add(object_id)

        if hasattr(object_, "feature_names_in_"):
            return list(object_.feature_names_in_)

        for attribute in [
            "regressor_",
            "regressor",
            "estimator_",
            "estimator",
            "transformer_",
            "transformer",
        ]:
            nested_object = getattr(
                object_,
                attribute,
                None,
            )

            result = walk(nested_object)

            if result is not None:
                return result

        if hasattr(object_, "steps"):
            for _, nested_object in object_.steps:
                result = walk(nested_object)

                if result is not None:
                    return result

        if hasattr(object_, "named_steps"):
            for nested_object in object_.named_steps.values():
                result = walk(nested_object)

                if result is not None:
                    return result

        return None

    return walk(model)


# ==============================================================
# RIDGE
# ==============================================================

ridge_expected_columns = find_feature_names_in(
    ridge_final_model
)

if ridge_expected_columns is None:
    raise ValueError(
        "Не удалось определить колонки Ridge."
    )

ridge_missing_columns = [
    column
    for column in ridge_expected_columns
    if column not in X_test_features.columns
]

if ridge_missing_columns:
    raise KeyError(
        f"Ridge: отсутствуют колонки: {ridge_missing_columns}"
    )

X_test_ridge = sanitize_sklearn_frame(
    X_test_features[ridge_expected_columns]
)

ridge_test_pred = np.maximum(
    ridge_final_model.predict(X_test_ridge),
    1,
)

print(
    "Ridge prediction shape:",
    ridge_test_pred.shape,
)


# ==============================================================
# TEXT RIDGE
# ==============================================================

# Text-pipeline самостоятельно выбирает нужные текстовые колонки.
# Поэтому передаём полный безопасный feature frame.
X_test_text = sanitize_sklearn_frame(
    X_test_features
)

text_test_pred = np.maximum(
    text_final_model.predict(X_test_text),
    1,
)

print(
    "Text Ridge prediction shape:",
    text_test_pred.shape,
)


# ==============================================================
# ПРОВЕРКА ШЕСТИ ПРОГНОЗОВ
# ==============================================================

component_predictions = pd.DataFrame(
    {
        "ridge": ridge_test_pred,
        "v5": v5_test_pred,
        "v6": v6_test_pred,
        "v7": v7_test_pred,
        "stats": stats_test_pred,
        "text": text_test_pred,
    }
)

assert len(component_predictions) == 8341
assert component_predictions.notna().all().all()
assert (component_predictions > 0).all().all()

display(component_predictions.describe())
display(component_predictions.corr().round(5))


# ==============================================================
# ТОЧНЫЙ SIX-MODEL ENSEMBLE V3
# ==============================================================

FINAL_WEIGHTS_V3 = {
    "ridge": 0.237658,
    "v5": 0.106625,
    "v6": 0.083058,
    "v7": 0.197047,
    "stats": 0.309853,
    "text": 0.065759,
}

GLOBAL_CALIBRATION = 0.98

assert np.isclose(
    sum(FINAL_WEIGHTS_V3.values()),
    1.0,
)

raw_v3_prediction = sum(
    component_predictions[column].to_numpy() * weight
    for column, weight in FINAL_WEIGHTS_V3.items()
)

final_v3_prediction = np.maximum(
    raw_v3_prediction * GLOBAL_CALIBRATION,
    1,
)

final_submission = pd.DataFrame(
    {
        "Цена": final_v3_prediction,
    }
)

OUTPUT_CSV_PATH = (
    SUBMISSION_DIR
    / "submission_exact_v3_six_models_0_98.csv"
)

OUTPUT_ZIP_PATH = (
    SUBMISSION_DIR
    / "submission_exact_v3_six_models_0_98.zip"
)

final_submission.to_csv(
    OUTPUT_CSV_PATH,
    index=False,
)

with zipfile.ZipFile(
    OUTPUT_ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(
        OUTPUT_CSV_PATH,
        arcname="submission.csv",
    )

with zipfile.ZipFile(
    OUTPUT_ZIP_PATH,
    mode="r",
) as archive:
    print("Файлы в архиве:", archive.namelist())

assert zipfile.ZipFile(
    OUTPUT_ZIP_PATH
).namelist() == ["submission.csv"]

print("\nCSV:", OUTPUT_CSV_PATH)
print("ZIP:", OUTPUT_ZIP_PATH)

display(final_submission.head())
display(final_submission.describe())

Ridge prediction shape: (8341,)


AttributeError: 'dict' object has no attribute 'predict'